# Transfer Learning with Pretrained CNNs

This notebook explores several transfer-learning workflows:

1. Inspecting a pretrained ResNet50.
2. Extracting image features with VGG16.
3. Partially unfreezing an InceptionV3 backbone.
4. Fine-tuning a small binary classifier with ResNet50.
5. Building a frozen InceptionV3 classifier.

The examples use small subsets where possible so they can be tested quickly.

In [ ]:

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications import ResNet50

tf.keras.utils.set_random_seed(23)

resnet = ResNet50(
    include_top=True,
    weights="imagenet"
)

print(f"ResNet50 layers: {len(resnet.layers)}")
print(f"Trainable layers: {sum(int(layer.trainable) for layer in resnet.layers)}")
resnet.summary()


## Feature extraction with VGG16

CIFAR-10 images are resized to the input size expected by VGG16. The original classifier is removed, so the network returns convolutional feature maps.

In [ ]:

from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

(train_images, train_labels), _ = keras.datasets.cifar10.load_data()

sample_batch = train_images[20:28].astype("float32")
sample_batch = tf.image.resize(sample_batch, [224, 224])
sample_batch = preprocess_input(sample_batch)

vgg_backbone = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

feature_maps = vgg_backbone(sample_batch, training=False)

print("Input batch:", sample_batch.shape)
print("VGG16 feature maps:", feature_maps.shape)


## Controlled fine-tuning with InceptionV3

The convolutional base is initially frozen. Only the final 24 layers are made trainable, while a new five-class prediction head is attached.

In [ ]:

from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras import Model

inception_base = InceptionV3(
    weights="imagenet",
    include_top=False,
    input_shape=(299, 299, 3)
)

inception_base.trainable = False

for block in inception_base.layers[-24:]:
    block.trainable = True

pooled = GlobalAveragePooling2D()(inception_base.output)
predictions = Dense(5, activation="softmax", name="class_output")(pooled)

inception_classifier = Model(
    inputs=inception_base.input,
    outputs=predictions
)

trainable_count = sum(
    tf.keras.backend.count_params(variable)
    for variable in inception_classifier.trainable_weights
)

print("Trainable parameters:", trainable_count)


## Small binary classification experiment with ResNet50

Only CIFAR-10 classes `0` and `1` are retained. The images are resized to ResNet50's expected input dimensions. A compact subset is used to keep the demonstration manageable.

In [ ]:

from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.layers import Dropout

(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

train_ids = tf.reshape(y_train, [-1])
test_ids = tf.reshape(y_test, [-1])

keep_train = tf.logical_or(train_ids == 0, train_ids == 1)
keep_test = tf.logical_or(test_ids == 0, test_ids == 1)

x_train = x_train[keep_train.numpy()][:800]
y_train = train_ids.numpy()[keep_train.numpy()][:800]
x_test = x_test[keep_test.numpy()][:200]
y_test = test_ids.numpy()[keep_test.numpy()][:200]

x_train = preprocess_input(tf.image.resize(x_train, [224, 224]))
x_test = preprocess_input(tf.image.resize(x_test, [224, 224]))

binary_base = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)
binary_base.trainable = False

head_input = binary_base.output
head_output = GlobalAveragePooling2D()(head_input)
head_output = Dropout(0.35)(head_output)
head_output = Dense(1, activation="sigmoid")(head_output)

binary_model = Model(binary_base.input, head_output)

binary_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=2e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history = binary_model.fit(
    x_train,
    y_train,
    validation_split=0.15,
    epochs=2,
    batch_size=32,
    verbose=1
)


## InceptionV3 as a frozen binary classifier

This final model uses a frozen InceptionV3 feature extractor and a sigmoid output for binary prediction.

In [ ]:

frozen_inception = InceptionV3(
    weights="imagenet",
    include_top=False,
    input_shape=(299, 299, 3)
)
frozen_inception.trainable = False

classifier_input = frozen_inception.input
classifier_features = GlobalAveragePooling2D()(frozen_inception.output)
classifier_features = Dropout(0.4)(classifier_features)
classifier_output = Dense(1, activation="sigmoid")(classifier_features)

frozen_binary_model = Model(
    classifier_input,
    classifier_output,
    name="frozen_inception_binary_model"
)

frozen_binary_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

frozen_binary_model.summary()


### Notes

- Pretrained weights require internet access the first time they are downloaded.
- Resizing small images does not add new visual information; it only matches the backbone input shape.
- Batch-normalization layers require care during fine-tuning.
- For a real project, use a larger dataset and evaluate on a held-out test set.